In [ ]:
import pandas as pd
from datetime import datetime


In [ ]:
def set_ex_id(ex_id:int) -> str:
    if ex_id < 10:
        ex_id_str = f'0{ex_id}'
    else:
        ex_id_str = str(ex_id)
    return ex_id_str

In [ ]:

df_hydro = pd.read_excel("/Users/6256481/Code/river-hierarchy/SWORD_gauge_match/outputs/mask_hydro_quality.xlsx", sheet_name = 'Hydrograph_quality')
df_hydro = df_hydro[(~df_hydro['Peak_quality'].isin(['Bad', '-', 'Medium'
''])) & (~df_hydro['Peak_match'].isin(['Bad', '-']))]
df_hydro = df_hydro[['Example', 'Up', 'Period_start', 'Period_end', 'Orientation', 'Forcing_time'
                     ,'mask_file', 'min_celerity', 'max_celerity', 'panelized']]

# mask prepping

In [ ]:
ex_id = 55
row = df_hydro[df_hydro['Example'] == ex_id].iloc[0]

if ex_id < 10:
  ex_id_str = f'0{ex_id}'
else:
  ex_id_str = str(ex_id)
################################################
# Mask prep: Create binary mask from selected mask
################################################
mask_prep_string = rf"""/opt/anaconda3/envs/river-hierarchy-rivgraph/bin/python -B \
  rivgraph_centerline/run_workflow.py prepare-mask \
  --name {ex_id_str} \
  --source-mask /Volumes/PhD/river_hierarchy/data/water_masks/{row['mask_file']}"""
print("Mask prep: create binary mask with correct crs\n")
print(mask_prep_string)
print('\n\n')

PYTHON = "/opt/anaconda3/envs/river-hierarchy-rivgraph/bin/python"
import subprocess

def run_cmd(label, cmd, env=None):
        print(f"\n{'=' * 80}")
        print(label)
        print(" ".join(map(str, cmd)))
        print(f"{'=' * 80}\n")

        subprocess.run(
            cmd,
            check=True,
            env=env,
        )

mask_prep_cmd = [
    PYTHON,
    "-B",
    "rivgraph_centerline/run_workflow.py",
    "prepare-mask",
    "--name", ex_id_str,
    "--source-mask", f"/Volumes/PhD/river_hierarchy/data/water_masks/{row['mask_file']}",
    "--threshold", "20"
]
run_cmd("mask prepping", mask_prep_cmd)

# Apply edits and run Rivgraph

In [ ]:
def app_edits_run_rivgraph(ex_id):
    row = df_hydro[df_hydro['Example'] == ex_id].iloc[0]

    if ex_id < 10:
        ex_id = f'0{ex_id}'
    mask_id = ex_id

    orientation = row['Orientation']

    # create base centerline file
    ################################################
    edits_run_rivgraph = rf"""/opt/anaconda3/envs/river-hierarchy-rivgraph/bin/python -B \
    rivgraph_centerline/run_workflow.py apply-edits \
    --name {ex_id} \
    --keep-largest-water-component \
        
    MPLCONFIGDIR=/tmp/matplotlib-rivgraph \
    /opt/anaconda3/envs/river-hierarchy-rivgraph/bin/python -B \
    rivgraph_centerline/run_workflow.py run-rivgraph \
    --name {ex_id} \
    --exit-sides {orientation} \
    --run-dir rivgraph_centerline/outputs/runs/{ex_id} \
    --assign-flow \
    --flow-mesh-mode auto"""
    print(edits_run_rivgraph)
    print('\n\n')

    # run_width_check_GUI_cmd = rf"""/opt/anaconda3/envs/river-hierarchy-rivgraph/bin/python \
    # /opt/anaconda3/envs/river-hierarchy-rivgraph/bin/python \
    # network_variants/width_review_gui.py \
    # /Users/6256481/Code/river-hierarchy/network_variants/output/{ex_id}_indep__anchored/states/S000_base/variant"""
    # print(run_width_check_GUI_cmd)
    # print('\n\n')

ex_id = 5
app_edits_run_rivgraph(ex_id)

# Change directions

In [ ]:
ex_id = 37

row = df_hydro[df_hydro['Example'] == ex_id].iloc[0]
orientation = row['Orientation']
if ex_id < 10:
    ex_id = f'0{ex_id}'

direction_cleanup_cmd = rf"""/opt/anaconda3/envs/river-hierarchy-rivgraph/bin/python \
  hierarchy_level_definition/manual_review/apply_link_reversals.py \
  --csv /Users/6256481/Code/river-hierarchy/rivgraph_centerline/outputs/runs/flow_direction_adjustments.csv \
  --example-id {ex_id} \
  --review-root /Users/6256481/Code/river-hierarchy/rivgraph_centerline/outputs/runs \
  --validation-root /Users/6256481/Code/river-hierarchy/hierarchy_level_definition/graph_building/outputs \
  --summary-json /Users/6256481/Code/river-hierarchy/rivgraph_centerline/outputs/runs/flow_direction_adjustments_summary.json \
  --old-backup-mode use       
"""
#overwrite for first time use and "use" for subsequent uses
# overwrite / use

print(direction_cleanup_cmd)

# Run Base only

In [ ]:
ex_id = 56

row = df_hydro[df_hydro['Example'] == ex_id].iloc[0]
orientation = row['Orientation']
if ex_id < 10:
    ex_id = f'0{ex_id}'
if row['panelized'] == True:
    panelized = 'panelized'
else:
    panelized = 'full_raster'

print(rf"""/opt/anaconda3/envs/river-hierarchy-rivgraph/bin/python \
  network_variants/run_collapse_experiment.py \
  independent-units \
  --cleaned-mask rivgraph_centerline/outputs/runs/{ex_id}/masks_cleaned/{ex_id}_cleaned.tif \
  --reviewed-links rivgraph_centerline/outputs/runs/{ex_id}/rivgraph/{ex_id}_links.gpkg \
  --reviewed-nodes rivgraph_centerline/outputs/runs/{ex_id}/rivgraph/{ex_id}_nodes.gpkg \
  --exit-sides {orientation} \
  --experiment-id {ex_id}_indep \
  --output-dir network_variants/output/{ex_id}_indep \
  --collapse-corridor-mode anchored \
  --base-only \
  --execution-mode {panelized}
""")

# width adjustment GUI

In [ ]:
ex_id = 37

row = df_hydro[df_hydro['Example'] == ex_id].iloc[0]

if ex_id < 10:
    ex_id = f'0{ex_id}'
print(rf"""/opt/anaconda3/envs/river-hierarchy-rivgraph/bin/python \
  network_variants/width_review_gui.py \
  /Users/6256481/Code/river-hierarchy/network_variants/output/{ex_id}_indep__anchored/states/S000_base/variant \
  --dt-points \
  --overwrite-backup
""")
  # --start-over Add start over if review needs to be done again


# Panelization for large sections

In [ ]:
ex_id = 46
row = df_hydro[df_hydro['Example'] == ex_id].iloc[0]

ex_id_str = set_ex_id(ex_id)
print("Create provisional panel breaks for panelized rivgraph run\n")
print(rf"""/opt/anaconda3/envs/river-hierarchy-rivgraph/bin/python -B \
rivgraph_centerline/run_workflow.py create-panel-breaks \
  --name {ex_id_str} \
  --run-dir rivgraph_centerline/outputs/runs/{ex_id_str} \
  --max-panel-pixels 120000000 \
  --overwrite""")

print('\n\n')
print("Plan panels for panelized rivgraph run\n")
print(rf"""/opt/anaconda3/envs/river-hierarchy-rivgraph/bin/python -B \
  rivgraph_centerline/run_workflow.py plan-panels \
  --name {ex_id_str} \
  --run-dir rivgraph_centerline/outputs/runs/{ex_id_str} \
  --mask rivgraph_centerline/outputs/runs/{ex_id_str}/masks_prepared/{ex_id_str}_binary_projected.tif \
  --overwrite
  """)

print('\n\n')
print("Apply panelized edits for panelized rivgraph run\n")
print(rf"""/opt/anaconda3/envs/river-hierarchy-rivgraph/bin/python -B \
  rivgraph_centerline/run_workflow.py apply-panelized-edits \
  --name {ex_id_str} --run-dir rivgraph_centerline/outputs/runs/{ex_id_str} \
  --keep-largest-water-component \
  --overwrite
  """)


print('\n\n')
print("Run panelized rivgraph for panelized rivgraph run\n")
print(rf"""MPLCONFIGDIR=/tmp/matplotlib-rivgraph \
/opt/anaconda3/envs/river-hierarchy-rivgraph/bin/python -B \
rivgraph_centerline/run_workflow.py run-panelized-rivgraph \
  --name {ex_id_str} \
  --run-dir rivgraph_centerline/outputs/runs/{ex_id_str} \
  --exit-sides {row['Orientation']} \
  --overwrite""")


# Collapse

In [ ]:
import pandas as pd

example_number = 46
row = df_hydro.loc[df_hydro["Example"].astype(int).eq(example_number)].iloc[0]

ex_id = f"{example_number:02d}"
orientation = row["Orientation"]


def change_timestamp(value):
    
    dt = pd.Timestamp(value)

    if dt.tzinfo is None:
        dt = dt.tz_localize("UTC")
    else:
        dt = dt.tz_convert("UTC")

    return dt.strftime("%Y-%m-%dT%H:%M:%SZ")


def run_commands():
    station_key = row["Up"]
    start_time = change_timestamp(row["Period_start"])
    end_time = change_timestamp(row["Period_end"])
    forcing_time = int(row["Forcing_time"])

    local = True
    panelized = bool(row["panelized"])

    if panelized and local:
        execution_mode = "panelized local_patch"
    elif panelized:
        execution_mode = "panelized"
    elif local:
        execution_mode = "local_patch"
    else:
        execution_mode = "full_raster"

    commands = []

    configurations = [
        {
            "mode": "independent-units",
            "experiment_suffix": "independent_units",
            "output_suffix": "indep",
            "base_state_source": None,
        },
        {
            "mode": "compound-bubbles",
            "experiment_suffix": "compound_bubbles",
            "output_suffix": "compound_bubbles",
            "base_state_source": (
                f"network_variants/output/{ex_id}_indep__anchored"
            ),
        },
    ]

    for config in configurations:
        arguments = [
            "/opt/anaconda3/envs/river-hierarchy-rivgraph/bin/python",
            "network_variants/run_collapse_experiment.py",
            config["mode"],
            (
                f"--cleaned-mask "
                f"rivgraph_centerline/outputs/runs/{ex_id}/"
                f"masks_cleaned/{ex_id}_cleaned.tif"
            ),
            (
                f"--reviewed-links "
                f"/Users/6256481/Code/river-hierarchy/"
                f"rivgraph_centerline/outputs/runs/{ex_id}/"
                f"rivgraph/{ex_id}_links.gpkg"
            ),
            (
                f"--reviewed-nodes "
                f"/Users/6256481/Code/river-hierarchy/"
                f"rivgraph_centerline/outputs/runs/{ex_id}/"
                f"rivgraph/{ex_id}_nodes.gpkg"
            ),
            f"--exit-sides {orientation}",
            (
                f"--experiment-id "
                f"{ex_id}_{config['experiment_suffix']}"
            ),
            (
                f"--output-dir "
                f"network_variants/output/"
                f"{ex_id}_{config['output_suffix']}"
            ),
            "--sword-node-source /Volumes/PhD/SWORD/v17c/",
            (
                "--sword-example-stations-source "
                "SWORD_gauge_match/outputs/"
                "hierarchy_examples_filtered_subdaily_"
                "manual_updates_final.gpkg"
            ),
            (
                "--sword-station-match-source "
                "SWORD_gauge_match/outputs/"
                "selected_event_stations_same_main_path.gpkg"
            ),
            "--sword-reach-buffer-steps 2",
            "--sword-wse-field wse_obs_p50",
            "--slope-profile-mode reference_section",
            "--min-slope 1e-6",
            "--max-slope-for-k 0.009",
            "--section-slope-ratio-max 5",
            "--equivalent-length-field len",
            "--compute-equivalent-resistance",
            "--equivalent-resistance-law chezy",
            "--equivalent-geometry-method wide",
            "--equivalent-chezy-cz 22",
            "--equivalent-kb-mode based_reference_section",
            (
                "--equivalent-forcing-path "
                f"SWORD_gauge_match/outputs/subdaily_values/"
                f"{station_key[:2]}/subdaily_timeseries.parquet"
            ),
            f"--equivalent-forcing-station-key {station_key}",
            "--equivalent-station-key-column station_key",
            "--equivalent-time-column time",
            "--equivalent-discharge-column discharge",
            f"--equivalent-forcing-start-time {start_time}",
            f"--equivalent-forcing-end-time {end_time}",
            (
                "--equivalent-forcing-resample-seconds "
                f"{forcing_time}"
            ),
            (
                "--equivalent-forcing-output-cache-dir "
                "SWORD_gauge_match/outputs/rapid_forcing_cache"
            ),
            (
                "--equivalent-kb-model-path "
                "RAPID/assets/based_model_v2.ubj"
            ),
            (
                "--equivalent-kb-width-sample-field "
                "wid_adj_routing"
            ),
            "--collapse-corridor-mode anchored",
            "--equivalent-solver-backend energy_sparse",
            f"--execution-mode {execution_mode}",
        ]

        if config["base_state_source"] is not None:
            arguments.append(
                f"--base-state-source "
                f"{config['base_state_source']}"
            )

        slash = "\\"
        command = f" {slash}\n      ".join(arguments)
        commands.append(command)

    # Compound starts only if independent exits successfully.
    slash = "\\"
    print(f" && {slash}\n".join(commands))


run_commands()

## Island only collapse workflow
#### Always run after indep has been run

In [ ]:
# example_number = 2
# ex_id = f"{example_number:02d}"
# print(rf"""/opt/anaconda3/envs/river-hierarchy-rivgraph/bin/python -B \
#   network_variants/run_prepare_island_setup.py \
#   /Volumes/PhD/river_hierarchy/network_variants/output/{ex_id}_indep__anchored \
#   --print-candidates""")


example_numbers = [46]

commands = []

for example_number in example_numbers:
    ex_id = f"{int(example_number):02d}"
    commands.append(rf"""/opt/anaconda3/envs/river-hierarchy-rivgraph/bin/python -B \
  network_variants/run_prepare_island_setup.py \
  /Volumes/PhD/river_hierarchy/network_variants/output/{ex_id}_indep__anchored \
  --print-candidates""")

print("\n\n".join(commands))

In [ ]:
import pandas as pd

example_number = 45
row = df_hydro.loc[df_hydro["Example"].astype(int).eq(example_number)].iloc[0]

ex_id = f"{example_number:02d}"
orientation = row["Orientation"]

# Direct local island workflow: no full-raster island state is created first.
local = True
# Set True only to resume a previously interrupted island experiment root.
resume_existing_experiment = False
# Set True only when adopting a pre-created metadata-only island root.
adopt_metadata_root = False

def run_island_command():
    base_experiment = (
        f"/Volumes/PhD/river_hierarchy/network_variants/output/"
        f"{ex_id}_indep__anchored"
    )
    reviewed_setup = f"{base_experiment}/island_only_setup_8connected"

    base_variant = f"{base_experiment}/states/S000_base/variant"

    execution_mode = "local_patch" if local else "full_raster"

    arguments = [
        "/opt/anaconda3/envs/river-hierarchy-rivgraph/bin/python",
        "-B",
        "network_variants/run_island_experiment.py",
        f"--reviewed-setup-dir {reviewed_setup}",
        f"--base-state-source {base_experiment}",
        (
            "--output-dir "
            f"/Volumes/PhD/river_hierarchy/network_variants/output/"
            f"{ex_id}_island__anchored"
        ),
        *(
            ["--resume-existing-experiment"]
            if resume_existing_experiment else
            (["--adopt-metadata-root"] if adopt_metadata_root else [])
        ),

        # Use the actual independent BASE products, not the original raw links.
        (
            "--cleaned-mask "
            f"{base_variant}/mask/{ex_id}__S000_base_collapsed.tif"
        ),
        (
            "--wet-reference-mask "
            f"{base_variant}/mask/{ex_id}__S000_base_collapsed.tif"
        ),
        (
            "--reviewed-links "
            f"{base_variant}/widths/links_with_width_families.gpkg"
        ),
        (
            "--reviewed-nodes "
            f"{base_variant}/directed/"
            f"{ex_id}__S000_base_directed_nodes.gpkg"
        ),
        (
            "--boundary-component-labels "
            f"{base_experiment}/boundary_components/"
            f"{ex_id}_full_extent_land_water_labels_with_snapped_buffered_border_cut.tif"
        ),

        f"--exit-sides {orientation}",
        f"--example-id {ex_id}",
        f"--execution-mode {execution_mode}",
        *(
            [
                "--local-patch-reference-centerline "
                f"rivgraph_centerline/outputs/runs/{ex_id}/rivgraph/{ex_id}_centerline.gpkg"
            ]
            if local else []
        ),

        "--sword-node-source /Volumes/PhD/SWORD/v17c/",
        "--sword-wse-field wse_obs_p50",
        (
            "--sword-example-stations-source "
            "SWORD_gauge_match/outputs/"
            "hierarchy_examples_filtered_subdaily_manual_updates_final.gpkg"
        ),
        (
            "--sword-station-match-source "
            "SWORD_gauge_match/outputs/"
            "selected_event_stations_same_main_path.gpkg"
        ),
        "--sword-reach-buffer-steps 2",

        # Reuses the independent experiment's stored resistance configuration.
        "--compute-equivalent-resistance",
    ]

    slash = "\\"
    print(f" {slash}\n      ".join(arguments))


run_island_command()

# RAPID prep and running

In [ ]:
# # Preserve completed cases when a later case fails and this cell is rerun.
# phase7_results = globals().get('phase7_results', {})
# if not RUN_PHASE7_BATCH:
#     print('Phase-7 batch not run. Set RUN_PHASE7_BATCH = True in the previous cell, then rerun this cell.')
# else:
#     for example_id, unit_id, exit_sides in PHASE7_CASES:
#         if (example_id, unit_id) in phase7_results:
#             print(f'Reusing completed Phase-7 result: example {example_id}, unit {unit_id}.')
#             continue
#         print(f'Running Phase 7: example {example_id}, unit {unit_id} ...')
#         phase7_results[(example_id, unit_id)] = run_phase7_validation_case(example_id, unit_id, exit_sides)

#     # Refresh acceptance decisions for cached results after a threshold or
#     # diagnostic definition changes; no RivGraph work is repeated.
#     for result_i in phase7_results.values():
#         metrics_i = result_i['metrics']
#         if 'max_changed_geometry_hausdorff_pixels' not in metrics_i:
#             with rasterio.open(result_i['variant'].output_paths['local_collapsed_mask']) as source:
#                 pixel_size_i = float(max(abs(source.transform.a), abs(source.transform.e)))
#             metrics_i['local_pixel_size_m'] = pixel_size_i
#             metrics_i['max_changed_geometry_hausdorff_pixels'] = metrics_i['max_changed_geometry_hausdorff_m'] / pixel_size_i
#         result_i['checks']['changed geometry matched one-to-one'] = (
#             metrics_i['nonunique_changed_geometry_matches'] == 0
#             and metrics_i['max_changed_geometry_hausdorff_pixels'] <= PHASE7_LIMITS['changed_geometry_hausdorff_pixels']
#         )
#         result_i['checks']['changed slopes reproduce full raster'] = (
#             metrics_i['max_changed_slope_relative_difference'] <= PHASE7_LIMITS['changed_slope_relative_difference']
#         )

#     check_rows = []
#     metric_rows = []
#     for (example_id, unit_id), result_i in phase7_results.items():
#         for check, passed in result_i['checks'].items():
#             check_rows.append({'example_id': example_id, 'unit_id': unit_id, 'check': check, 'passed': bool(passed)})
#         metric_rows.append({'example_id': example_id, 'unit_id': unit_id, **result_i['metrics'], 'scratch_output': str(result_i['root'])})
#     phase7_checks = pd.DataFrame(check_rows)
#     phase7_metrics = pd.DataFrame(metric_rows)
#     display(phase7_metrics)
#     display(phase7_checks.pivot(index='check', columns='unit_id', values='passed'))
#     phase7_all_checks_passed = bool(phase7_checks['passed'].all())
#     if phase7_all_checks_passed:
#         print('All Phase-7 acceptance checks passed.')
#     else:
#         print('One or more Phase-7 checks need review; continue to the maps below.')
#         display(phase7_checks.loc[~phase7_checks['passed']])

In [ ]:
ex_id = 46

row = df_hydro[df_hydro["Example"] == ex_id].iloc[0]

if ex_id < 10:
    ex_id = f"0{ex_id}"

orientation = row["Orientation"]


def change_timestamp(dt_str):
    dt = datetime.strptime(dt_str, "%Y-%m-%d %H:%M:%S%z")
    return dt.strftime("%Y-%m-%dT%H:%M:%SZ")


def run_commands():
    station_key = row["Up"]
    start_time = change_timestamp(row["Period_start"])
    end_time = change_timestamp(row["Period_end"])
    forcing_time = int(row["Forcing_time"])
    min_celerity = row["min_celerity"]
    max_celerity = row["max_celerity"]
    rapid_work_root = "/Volumes/PhD/river_hierarchy/network_variants/rapid_work"

    commands = []

    for nvs in ["indep", "compound_bubbles"]:
        variant_dir = (
            f"/Users/6256481/Code/river-hierarchy/"
            f"network_variants/output/{ex_id}_{nvs}__anchored"
        )

        # 1. Recompute state-level equivalent resistance and create
        #    the experiment-level aggregate comparison tables.
        k_equivalent = rf"""/opt/anaconda3/envs/river-hierarchy-rivgraph/bin/python -B \
  network_variants/run_equivalent_resistance_experiment.py \
  {variant_dir} \
  --recompute-state-hierarchy \
  --reuse-manifest-config"""

        commands.append(k_equivalent)

        # 2. Prepare all networks for RAPID.
        rapid_prep = rf"""/opt/anaconda3/envs/river-hierarchy-rivgraph/bin/python \
  RAPID/run_prepare_experiment.py \
  {variant_dir} \
  --work-root {rapid_work_root} \
  --forcing-path SWORD_gauge_match/outputs/subdaily_values/{station_key[0:2]}/subdaily_timeseries.parquet \
  --forcing-station-key {station_key} \
  --station-key-column station_key \
  --time-column time \
  --discharge-column discharge \
  --forcing-start-time {start_time} \
  --forcing-end-time {end_time} \
  --forcing-resample-seconds {forcing_time} \
  --forcing-output-cache-dir SWORD_gauge_match/outputs/rapid_forcing_cache \
  --kb-mode based_reference_section \
  --kb-model-path RAPID/assets/based_model_v2.ubj \
  --width-field wid_adj_routing \
  --kb-width-sample-field wid_adj_routing \
  --use-celerity-capping \
  --min-celerity-mps {min_celerity} \
  --max-celerity-mps {max_celerity} \
  --use-stability-effective-length-cap \
  --stability-effective-length-cap-margin-seconds 10 \
  --stability-effective-length-cap-celerity-rel-buffer 0.03 \
  --preferred-routing-dt-seconds {forcing_time} \
  --forcing-dt-search-max-offset-seconds 120 \
  --target-subreach-length-m 200 \
  --min-effective-length-for-k-m 200 \
  --auto-spinup-from-k \
  --spinup-k-multiplier 1.1"""

        commands.append(rapid_prep)

        # 3. Execute RAPID.
        # Successful states retain compact analysis products; failed states retain prep diagnostics.
        rapid_run = rf"""/opt/anaconda3/envs/river-hierarchy-rivgraph/bin/python \
  RAPID/run_rapid_experiment.py \
  {variant_dir} \
  --compact-qout \
  --write-mapping-parquet \
  --drop-prepared-inputs"""

        commands.append(rapid_run)

    # Stop immediately if any preceding command fails.
    print(" && \\\n".join(commands))


run_commands()

In [ ]:
example_numbers = [15]

python = "/opt/anaconda3/envs/river-hierarchy-rivgraph/bin/python"
output_root = "/Volumes/PhD/river_hierarchy/network_variants/output"
rapid_work_root = "/Volumes/PhD/river_hierarchy/network_variants/rapid_work"

commands = []

for example_number in example_numbers:
    ex_id = f"{int(example_number):02d}"

    # Example 46 was intentionally retained in the v2 island root.
    island_name = (
        f"{ex_id}_island__anchored"
        if ex_id == "46"
        else f"{ex_id}_island__anchored"
    )

    island_dir = f"{output_root}/{island_name}"
    base_dir = f"{output_root}/{ex_id}_indep__anchored"
    work_dir = f"{rapid_work_root}/{island_name}"

    prepare = rf"""{python} -B \
RAPID/run_prepare_island_observations.py \
{island_dir}/island_analysis_manifest_strict.csv \
{island_dir}/island_hydraulic_context_registry.csv \
{base_dir} \
--work-root {work_dir}"""

    run = rf"""{python} -B \
RAPID/run_island_observations.py \
{island_dir}/island_rapid_prep_registry.csv \
--compact-qout \
--write-mapping-parquet \
--drop-prepared-inputs"""

    compare = rf"""{python} -B \
RAPID/compare_island_observations.py \
{island_dir}/island_analysis_manifest_strict.csv \
{base_dir}"""

    commands.append(" && \\\n".join([prepare, run, compare]))

print("\n\n".join(commands))

# Add centerline files to QGIS

In [ ]:
exid = 22
for exid in [22,37,41, 45,46,48, 54,55, 56]:
print(rf"""ROOT="/home/6256481/River_Morphology/river_hierarchy"

cd "$ROOT/code"
ALLOW_OUTPUT_RESET=1 \
  ./scripts/run_hpc_science_experiments.sh {exid} \
  > "$ROOT/logs/{exid}_science_launcher.log" 2>&1 &

disown""")

In [ ]:
for exid in [22, 37, 41, 45, 46, 54, 55, 56]:
    print(f'''ROOT="/home/6256481/River_Morphology/river_hierarchy"

cd "$ROOT/code"
./scripts/run_hpc_science_experiments.sh {exid} \
  > "$ROOT/logs/{exid}_science_launcher.log" 2>&1 &

disown
''')

In [ ]:
from glob import glob
from pathlib import Path
import numpy as np
ex = '02'
# ut = f'{ex}_compound_bubbles__anchored'
ut = f'{ex}_indep__anchored'
base = f"/Volumes/PhD/river_hierarchy/network_variants/output/{ut}/states/"



gpkg_paths = glob(base + "**/variant/directed/*directed_links.gpkg*", recursive = True)
def sort_key(path):
    token = path.split('_')[-3]
    if token == "base":
        return 0
    return int(token) + 1

gpkg_paths_sorted = sorted(gpkg_paths, key=sort_key)
gpkg_paths_sorted

In [ ]:
from qgis.core import QgsVectorLayer, QgsProject
import os

gpkg_paths =['/Volumes/PhD/river_hierarchy/network_variants/output/11_indep__anchored/states/S000_base/variant/directed/11__S000_base_directed_links.gpkg',
 '/Volumes/PhD/river_hierarchy/network_variants/output/11_indep__anchored/states/S009_unit_1/variant/directed/11__S009_unit_1_directed_links.gpkg',
 '/Volumes/PhD/river_hierarchy/network_variants/output/11_indep__anchored/states/S016_unit_2/variant/directed/11__S016_unit_2_directed_links.gpkg',
 '/Volumes/PhD/river_hierarchy/network_variants/output/11_indep__anchored/states/S020_unit_3/variant/directed/11__S020_unit_3_directed_links.gpkg',
 '/Volumes/PhD/river_hierarchy/network_variants/output/11_indep__anchored/states/S001_unit_4/variant/directed/11__S001_unit_4_directed_links.gpkg',
 '/Volumes/PhD/river_hierarchy/network_variants/output/11_indep__anchored/states/S007_unit_5/variant/directed/11__S007_unit_5_directed_links.gpkg',
 '/Volumes/PhD/river_hierarchy/network_variants/output/11_indep__anchored/states/S019_unit_6/variant/directed/11__S019_unit_6_directed_links.gpkg',
 '/Volumes/PhD/river_hierarchy/network_variants/output/11_indep__anchored/states/S015_unit_7/variant/directed/11__S015_unit_7_directed_links.gpkg',
 '/Volumes/PhD/river_hierarchy/network_variants/output/11_indep__anchored/states/S012_unit_8/variant/directed/11__S012_unit_8_directed_links.gpkg',
 '/Volumes/PhD/river_hierarchy/network_variants/output/11_indep__anchored/states/S013_unit_9/variant/directed/11__S013_unit_9_directed_links.gpkg',
 '/Volumes/PhD/river_hierarchy/network_variants/output/11_indep__anchored/states/S010_unit_10/variant/directed/11__S010_unit_10_directed_links.gpkg',
 '/Volumes/PhD/river_hierarchy/network_variants/output/11_indep__anchored/states/S008_unit_11/variant/directed/11__S008_unit_11_directed_links.gpkg',
 '/Volumes/PhD/river_hierarchy/network_variants/output/11_indep__anchored/states/S005_unit_12/variant/directed/11__S005_unit_12_directed_links.gpkg',
 '/Volumes/PhD/river_hierarchy/network_variants/output/11_indep__anchored/states/S014_unit_13/variant/directed/11__S014_unit_13_directed_links.gpkg',
 '/Volumes/PhD/river_hierarchy/network_variants/output/11_indep__anchored/states/S017_unit_14/variant/directed/11__S017_unit_14_directed_links.gpkg',
 '/Volumes/PhD/river_hierarchy/network_variants/output/11_indep__anchored/states/S006_unit_15/variant/directed/11__S006_unit_15_directed_links.gpkg',
 '/Volumes/PhD/river_hierarchy/network_variants/output/11_indep__anchored/states/S018_unit_16/variant/directed/11__S018_unit_16_directed_links.gpkg',
 '/Volumes/PhD/river_hierarchy/network_variants/output/11_indep__anchored/states/S011_unit_17/variant/directed/11__S011_unit_17_directed_links.gpkg',
 '/Volumes/PhD/river_hierarchy/network_variants/output/11_indep__anchored/states/S003_unit_18/variant/directed/11__S003_unit_18_directed_links.gpkg',
 '/Volumes/PhD/river_hierarchy/network_variants/output/11_indep__anchored/states/S002_unit_19/variant/directed/11__S002_unit_19_directed_links.gpkg',
 '/Volumes/PhD/river_hierarchy/network_variants/output/11_indep__anchored/states/S004_unit_20/variant/directed/11__S004_unit_20_directed_links.gpkg']

project = QgsProject.instance()
root = project.layerTreeRoot()

for path in gpkg_paths:
    layer_name = os.path.splitext(os.path.basename(path))[0]
    layer = QgsVectorLayer(path, layer_name, "ogr")

    if layer.isValid():
        project.addMapLayer(layer)

        # Untick visibility in the Layers panel
        node = root.findLayer(layer.id())
        if node:
            node.setItemVisibilityChecked(False)

    else:
        print(f"Could not load: {path}")

In [ ]:
slope = d.groupby('id_link', as_index = False).first()
slope[slope['id_link'] == 81]['slope_used'].iloc[0]

In [ ]:
dd = d[d['rapid_celerity_capped'] == True]
dd['id_link'].unique()

In [ ]:
plt.scatter(d['id_link'], d['rapid_k'])

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

p = Path("/Users/6256481/Code/river-hierarchy/network_variants/output/21_indep__realized/states/S000_base/rapid/prep/rapid_link_attributes.csv")
df = pd.read_csv(p)

df["delta_h_used_m"] = df["slope_used"] * df["rapid_effective_length_m"]
df["delta_h_raw_m"] = df["raw_slope"] * df["rapid_effective_length_m"]
df["delta_h_section_ref_m"] = df["slope_section_ref"] * df["rapid_effective_length_m"]
df["slope_to_section_ratio"] = df["slope_used"] / df["slope_section_ref"]

cols = [
    "id_link",
    "rapid_k",
    "rapid_celerity_mps",
    "rapid_width_m",
    "rapid_effective_length_m",
    "slope_used",
    "raw_slope",
    "slope_section_ref",
    "delta_h_used_m",
    "delta_h_raw_m",
    "delta_h_section_ref_m",
    "slope_to_section_ratio",
    "slope_source_method",
    "slope_reason",
    "rapid_celerity_capped",
]

worst = df.sort_values("rapid_k", ascending=False)[cols].head(30)
worst


In [ ]:
S = df.groupby('id_link').first()['slope_used']
S.max()/ S.min(),S.max(), S.min() 

In [ ]:
node_p = Path("/Users/6256481/Code/river-hierarchy/network_variants/output/21_indep__realized/states/S000_base/hydraulic_context/node_hydraulic_context.csv")
nodes = pd.read_csv(node_p)

nodes[[
    "id_node",
    "node_wse_raw",
    "rapid_wse_for_slope",
    "rapid_wse_interpolated",
    "rapid_wse_source_method",
    "rapid_wse_profile_residual",
    "node_network_dist_out_m",
]].head(30)



In [ ]:
plt.scatter(nodes['node_network_dist_out_m'], nodes['rapid_wse_for_slope'])

plt.scatter(nodes.loc[nodes['id_node'].isin([53,54]), 'node_network_dist_out_m'], 
            nodes.loc[nodes['id_node'].isin([53,54]),'rapid_wse_for_slope'])

In [ ]:
D = pd.read_csv('/Users/6256481/Code/river-hierarchy/network_variants/output/tmp_test_unit_7_anchored/matching/link_match.csv')
D.loc[D['child_id_link'] == 20, ['child_id_link', 'parent_id_link', 'parent_overlap_length', 'parent_overlap_fraction']]

# Python run commands

In [ ]:
import subprocess
from pathlib import Path
import os

env_rivgraph = os.environ.copy()
env_rivgraph["MPLCONFIGDIR"] = "/tmp/matplotlib-rivgraph"
PYTHON = "/opt/anaconda3/envs/river-hierarchy-rivgraph/bin/python"

def run_cmd(label, cmd, env=None):
        print(f"\n{'=' * 80}")
        print(label)
        print(" ".join(map(str, cmd)))
        print(f"{'=' * 80}\n")

        subprocess.run(
            cmd,
            check=True,
            env=env,
        )
def create_CLI_commands(ex_id, nv, anchor):
    row = df_hydro[df_hydro['Example'] == ex_id].iloc[0]

    if ex_id < 10:
        ex_id = f'0{ex_id}'
    mask_id = ex_id

    orientation = row['Orientation']

    if nv == 'compound-bubbles':
        nv_ = 'compound_bubbles'
        nvs = 'compound_bubbles'

    elif nv == 'independent-units':
        nv_ = 'independent_units'
        nvs = 'indep'

    if anchor == True:
        anchored = 'anchored'
    else:
        anchored = 'realized'

    def change_timestamp(dt_str):
        dt = datetime.strptime(dt_str, "%Y-%m-%d %H:%M:%S%z")
        return dt.strftime("%Y-%m-%dT%H:%M:%SZ")

    STATION_KEY = row['Up']
    start_time  = change_timestamp(row['Period_start'])
    end_time    = change_timestamp(row['Period_end'])
    forcing_time = int(row['Forcing_time'])

    PYTHON = "/opt/anaconda3/envs/river-hierarchy-rivgraph/bin/python"

    ################################################
    # Mask prep: Create binary mask from selected mask
    ################################################
    mask_prep_cmd = [
        PYTHON,
        "-B",
        "rivgraph_centerline/run_workflow.py",
        "prepare-mask",
        "--name", str(ex_id),
        "--source-mask", f"/Volumes/PhD/river_hierarchy/data/water_masks/{row['mask_file']}",
    ]

    ################################################
    # Apply edits from created polygons:
    # - polygon file: "example_name"_manual_edits
    # - add "action" field: commands = add, remove
    # - crs: crs of prepped file
    ################################################
    apply_edits_cmd = [
        PYTHON,
        "-B",
        "rivgraph_centerline/run_workflow.py",
        "apply-edits",
        "--name", str(ex_id),
    ]

    ################################################
    # Create base centerline file
    ################################################
    run_rivgraph_cmd = [
        PYTHON,
        "-B",
        "rivgraph_centerline/run_workflow.py",
        "run-rivgraph",
        "--name", str(ex_id),
        "--exit-sides", str(orientation),
        "--run-dir", f"rivgraph_centerline/outputs/runs/{ex_id}",
        "--assign-flow",
    ]

    ################################################
    # Unit detection
    ################################################
    level_detection_cmd = [
        PYTHON,
        "network_variants/run_collapse_experiment.py",
        str(nv),
        "--cleaned-mask", f"rivgraph_centerline/outputs/runs/{ex_id}/masks_cleaned/{ex_id}_cleaned.tif",
        "--reviewed-links", f"/Users/6256481/Code/river-hierarchy/rivgraph_centerline/outputs/runs/{ex_id}/rivgraph/{ex_id}_links.gpkg",
        "--reviewed-nodes", f"/Users/6256481/Code/river-hierarchy/rivgraph_centerline/outputs/runs/{ex_id}/rivgraph/{ex_id}_nodes.gpkg",
        "--exit-sides", str(orientation),
        "--experiment-id", f"{ex_id}_{nv_}",
        "--output-dir", f"network_variants/output/{ex_id}_{nvs}",
        "--sword-node-source", "/Volumes/PhD/SWORD/v17c/",
        "--sword-example-stations-source", "SWORD_gauge_match/outputs/hierarchy_examples_filtered_subdaily_manual_updates_final.gpkg",
        "--sword-station-match-source", "SWORD_gauge_match/outputs/selected_event_stations_same_main_path.gpkg",
        "--sword-reach-buffer-steps", "2",
        "--sword-wse-field", "wse_obs_p50",
        "--compute-equivalent-resistance",
        "--equivalent-resistance-law", "chezy",
        "--equivalent-geometry-method", "wide",
        "--equivalent-chezy-cz", "22",
        "--equivalent-kb-mode", "based_reference_section",
        "--equivalent-forcing-path", f"SWORD_gauge_match/outputs/subdaily_values/{STATION_KEY[0:2]}/subdaily_timeseries.parquet",
        "--equivalent-forcing-station-key", str(STATION_KEY),
        "--equivalent-station-key-column", "station_key",
        "--equivalent-time-column", "time",
        "--equivalent-discharge-column", "discharge",
        "--equivalent-forcing-start-time", str(start_time),
        "--equivalent-forcing-end-time", str(end_time),
        "--equivalent-forcing-resample-minutes", "18",
        "--equivalent-forcing-output-cache-dir", "SWORD_gauge_match/outputs/rapid_forcing_cache",
        "--equivalent-kb-model-path", "RAPID/assets/based_model_v2.ubj",
        "--equivalent-kb-width-sample-field", "width_wet",
        "--equivalent-kb-width-percentile", "90",
        "--collapse-corridor-mode", "both",
    ]

    ################################################
    # k_eq
    ################################################
    k_equivalent_cmd = [
        PYTHON,
        "network_variants/run_equivalent_resistance_experiment.py",
        str(variant_dir),

        "--equivalent-resistance-law", "chezy",
        "--equivalent-geometry-method", "wide",
        "--equivalent-chezy-cz", "22",
        "--equivalent-kb-mode", "based_reference_section",

        "--equivalent-forcing-path", f"SWORD_gauge_match/outputs/subdaily_values/{STATION_KEY[0:2]}/subdaily_timeseries.parquet",
        "--equivalent-forcing-station-key", str(STATION_KEY),
        "--equivalent-station-key-column", "station_key",
        "--equivalent-time-column", "time",
        "--equivalent-discharge-column", "discharge",

        "--equivalent-forcing-start-time", str(start_time),
        "--equivalent-forcing-end-time", str(end_time),
        "--equivalent-forcing-resample-minutes", str(forcing_time),
        "--equivalent-forcing-output-cache-dir", "SWORD_gauge_match/outputs/rapid_forcing_cache",

        "--equivalent-kb-model-path", "RAPID/assets/based_model_v2.ubj",
        "--equivalent-kb-width-sample-field", "width_wet",
        "--equivalent-kb-width-percentile", "90",
    ]

    ################################################
    # RAPID prep
    ################################################
    rapid_prep_cmd = [
        PYTHON,
        "RAPID/run_prepare_experiment.py",
        variant_dir,
        "--forcing-path", f"SWORD_gauge_match/outputs/subdaily_values/{STATION_KEY[0:2]}/subdaily_timeseries.parquet",
        "--forcing-station-key", STATION_KEY,
        "--station-key-column", "station_key",
        "--time-column", "time",
        "--discharge-column", "discharge",
        "--forcing-start-time", start_time,
        "--forcing-end-time", end_time,
        "--forcing-resample-minutes", str(forcing_time),
        "--forcing-output-cache-dir", "SWORD_gauge_match/outputs/rapid_forcing_cache",
        "--kb-mode", "based_reference_section",
        "--kb-model-path", "RAPID/assets/based_model_v2.ubj",
        "--kb-width-sample-field", "width_wet",
        "--kb-width-percentile", "90",
        "--use-celerity-capping",
        "--min-celerity-mps", "0.05",
        "--max-celerity-mps", "2.524",
        "--target-subreach-length-m", "200",
        "--min-effective-length-for-k-m", "200",
        "--max-slope-for-k", "0.005",
        "--section-slope-ratio-max", "5",
        "--vary-width-by-subreach",
        "--subreach-width-sample-field", "width_wet",
    ]

    ################################################
    # RAPID run
    ################################################
    rapid_run_cmd = [
        PYTHON,
        "RAPID/run_rapid_experiment.py",
        variant_dir,
    ]
    return (mask_prep_cmd, apply_edits_cmd, run_rivgraph_cmd, level_detection_cmd, 
            k_equivalent_cmd,rapid_prep_cmd, rapid_run_cmd)



In [ ]:
ex_id = 56
(mask_prep_cmd, apply_edits_cmd, run_rivgraph_cmd, level_detection_cmd, 
                k_equivalent_cmd,rapid_prep_cmd, rapid_run_cmd) = create_CLI_commands(ex_id, nv, anchor)

run_cmd("Mask prep: create binary mask with correct CRS", mask_prep_cmd)
run_cmd("Apply mask edits", apply_edits_cmd)
run_cmd("Run base rivgraph", run_rivgraph_cmd, env=env_rivgraph)

In [ ]:
ex_id = 56
nv = 'compound-bubbles' 
nv = 'independent-units'

for nv in ['compound-bubbles', 'independent-units']:
    (mask_prep_cmd, apply_edits_cmd, run_rivgraph_cmd, level_detection_cmd, 
                k_equivalent_cmd,rapid_prep_cmd, rapid_run_cmd) = create_CLI_commands(ex_id, nv, anchor)

    run_cmd("Mask prep: create binary mask with correct CRS", mask_prep_cmd)
    run_cmd("Apply mask edits", apply_edits_cmd)
    run_cmd("Run base rivgraph", run_rivgraph_cmd, env=env_rivgraph)
    run_cmd("Unit detection", level_detection_cmd)

In [ ]:
ex_id = 56
# nv = 'compound-bubbles' 
nv = 'independent-units'

anchor = True
for anchor in [True, False]:
    print(f'Anchor: {anchor}')
    (mask_prep_cmd, apply_edits_cmd, run_rivgraph_cmd, level_detection_cmd, 
                k_equivalent_cmd,rapid_prep_cmd, rapid_run_cmd) = create_CLI_commands(ex_id, nv, anchor)
    run_cmd("K equivalent", k_equivalent_cmd)
    run_cmd("RAPID prep", rapid_prep_cmd)
    run_cmd("RAPID run", rapid_run_cmd)